In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv(r"data\raw\HAM10000_metadata.csv")
counts = df['dx'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71' if c == 'nv' else '#e74c3c' for c in counts.index]
counts.plot(kind='bar', ax=ax1, color=colors)
ax1.set_title("Raw class counts\n(Green=majority, Red=minority)")
ax1.set_ylabel("Number of images")
ax1.tick_params(axis='x', rotation=45)

pct = (counts / len(df) * 100)
pct.plot(kind='bar', ax=ax2, color=colors)
ax2.set_title("Percentage of dataset")
ax2.set_ylabel("Percentage (%)")
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(r"notebooks\imbalance_chart.png")
plt.show()

print(f"Most common (NV):   {counts['nv']:4d} images = {counts['nv']/len(df)*100:.1f}%")
print(f"Least common (DF):  {counts['df']:4d} images = {counts['df']/len(df)*100:.1f}%")
print(f"Ratio: NV is {counts['nv']//counts['df']}x more common than DF")

In [ ]:
total = len(df)
nv_count = counts['nv']

dumb_accuracy = nv_count / total
print(f"Dumb model accuracy (always predict NV): {dumb_accuracy*100:.1f}%")
print(f"\nBut this model is USELESS for:")
print(f"  - Melanoma: would miss ALL {counts['mel']} cases")
print(f"  - DF: would miss ALL {counts['df']} cases")
print(f"  - VASC: would miss ALL {counts['vasc']} cases")
print(f"\nIn medicine, missing melanoma = patient dies.")
print(f"High overall accuracy means nothing if rare classes are missed.")

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

p = np.linspace(0.01, 0.99, 100)

ce_loss    = -np.log(p)
focal_g1   = -(1-p)**1 * np.log(p)
focal_g2   = -(1-p)**2 * np.log(p)
focal_g5   = -(1-p)**5 * np.log(p)

plt.figure(figsize=(10, 5))
plt.plot(p, ce_loss,  label='Cross-entropy (γ=0)', linewidth=2)
plt.plot(p, focal_g1, label='Focal loss γ=1', linewidth=2)
plt.plot(p, focal_g2, label='Focal loss γ=2 ← we use this', linewidth=2.5, color='red')
plt.plot(p, focal_g5, label='Focal loss γ=5', linewidth=2)
plt.xlabel("Predicted probability of correct class")
plt.ylabel("Loss")
plt.title("Focal Loss vs Cross-Entropy\nFocal loss reduces weight of easy examples")
plt.legend()
plt.ylim(0, 4)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(r"notebooks\focal_loss_comparison.png")
plt.show()

print("Key insight:")
print("At p=0.9 (easy example):")
print(f"  CE loss:    {-np.log(0.9):.3f}")
print(f"  Focal γ=2:  {-(1-0.9)**2 * np.log(0.9):.3f}  ← {(1-(1-0.9)**2 * np.log(0.9) / (-np.log(0.9)))*100:.0f}% reduction")
print(f"\nAt p=0.1 (hard example):")
print(f"  CE loss:    {-np.log(0.1):.3f}")
print(f"  Focal γ=2:  {-(1-0.1)**2 * np.log(0.1):.3f}  ← similar weight kept")

In [ ]:
epsilon = 0.1
num_classes = 7
hard_label = [0, 0, 1, 0, 0, 0, 0]

soft_label = [(1 - epsilon) * h + epsilon / num_classes for h in hard_label]

print("Hard label:", hard_label)
print("Soft label:", [round(x, 3) for x in soft_label])
print(f"\nCorrect class target: {soft_label[2]:.3f} instead of 1.0")
print(f"Other classes target: {soft_label[0]:.3f} instead of 0.0")
print(f"\nThis prevents the model from saying 'I am 100% sure this is NV'")
print(f"which would make it impossible to learn rare classes")